### Voici le notebook que nous mettons à disposition pour tester la pipeline de gestion du modèle

#### Le module est encapsulée dans le fichier model_main.py mais ce notebook permet de procéder séquentiellement pour que chaque tâche soit séparée et que l'on puisse analyser l'impact et l'utilité de chaque fonction 
#### Ce notebook simule l'apprentissage d'un modèle de production sur 1600 répertoires et l'apprentissage d'un modèle de staging sur tous les répertoires. Par la suite il permet de vérifier que l'on infère bien les répertoires qui n'ont pas leurs quatres représentations vectorielles. Enfin il illustre l'évaluation et le protocole de prise de décision qui suit l'évaluation.

##### Gestion des imports

In [1]:
import mlflow
import os
import pandas as pd
import numpy as np
import sys
import requests

from mlflow.exceptions import MlflowException
from apscheduler.schedulers.blocking import BlockingScheduler
from apscheduler.triggers.cron import CronTrigger
from pymongo import MongoClient

current = os.getcwd()
parent = os.path.dirname(current)
sys.path.append(parent)

from model.Doc2VecModel import Doc2VecModel
from model.Doc2VecTrainer import Doc2VecTrainer
from model.Evaluation import Evaluation
from model.MLFlowManager import MLFlowManager
from database.DatabaseManager import DatabaseManager

##### Définition de l'uri pour mlflow

In [2]:
mlflow_tracking_uri = os.path.join(parent, "artifacts", "mlruns")
os.environ["MLFLOW_TRACKING_URI"] = mlflow_tracking_uri

##### Définition de la surcouche mongodb

In [3]:
client = MongoClient('localhost', 27017)
database_manager = DatabaseManager(client['github'])

##### Définition des uri des modèles

In [4]:
staging_mlflow_model_readme_uri = 'models:/doc2vec_readme@staging'
staging_mlflow_model_others_uri = 'models:/doc2vec_others@staging'
    
production_mlflow_model_readme_uri = 'models:/doc2vec_readme@production'
production_mlflow_model_others_uri = 'models:/doc2vec_others@production'

##### Première étape: chargement des modèles en staging et en production courant

In [5]:
def check_if_model_exists(model_uri):
    """
    Vérifie si un modèle existe pour une uri donnée
    """
    try:
        model = mlflow.pyfunc.load_model(model_uri)
        return True
    except MlflowException as e:
        return False

In [17]:
def load_model(
    staging,
    is_model_registered,
    database_manager,
    mlflow_model_readme_uri,
    mlflow_model_others_uri
):
    """
    Charge les modèles selon les uri passés en paramètres
    Si aucun modèle ne correspond à l'uri on apprend un modèle pour cette uri
    """
    model = None
    
    train_df, test_df = database_manager.get_train_test_split_features()

    if is_model_registered:
        print(f"on charge le modèle de {staging}")
        model = Doc2VecModel(
            staging=staging,
            mlflow_readme_model_uri=mlflow_model_readme_uri,
            mlflow_others_model_uri=mlflow_model_others_uri,
        )
    else:
        # Sinon on l'apprend
        print(f"on doit apprendre le modèle de {staging}")
        doc2vec_trainer = Doc2VecTrainer(staging=staging)
        # Pour l'exemple on apprend le modèle de production sur 1600 répertoire et celui de staging sur tous les répertoires
        # Dans le module model_main.py on considère bien tous les répertoires disponibles au moment de l'apprentissage
        if staging == 'production':
            train_vectors, test_vectors = doc2vec_trainer.train(train_df.iloc[:1600], test_df.iloc[:1600])
        else:
            train_vectors, test_vectors = doc2vec_trainer.train(train_df, test_df)
        database_manager.insert_repos_vectors_for_one_stage(train_vectors, staging)
        database_manager.insert_repos_vectors_for_one_stage(test_vectors, staging)
        model = Doc2VecModel(
            staging=staging,
            mlflow_readme_model_uri=mlflow_model_readme_uri,
            mlflow_others_model_uri=mlflow_model_others_uri,
        )
        
    return model

In [7]:
is_staging_model_registered = check_if_model_exists(staging_mlflow_model_readme_uri)
is_production_model_registered = check_if_model_exists(production_mlflow_model_readme_uri)

In [8]:
staging_model = load_model(
    staging='staging',
    is_model_registered=is_staging_model_registered,
    database_manager=database_manager,
    mlflow_model_readme_uri=staging_mlflow_model_readme_uri,
    mlflow_model_others_uri=staging_mlflow_model_others_uri,
)

production_model = load_model(
    staging='production',
    is_model_registered=is_production_model_registered,
    database_manager=database_manager,
    mlflow_model_readme_uri=production_mlflow_model_readme_uri,
    mlflow_model_others_uri=production_mlflow_model_others_uri,
)

on doit apprendre le modèle


2025/03/11 04:04:23 INFO mlflow.models.signature: Inferring model signature from type hints
/home/choux/miniconda3/envs/aag-talp/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Successfully registered model 'doc2vec_readme'.
Created version '1' of model 'doc2vec_readme'.
2025/03/11 04:04:30 INFO mlflow.models.signature: Inferring model signature from type hints
Successfully registered model 'doc2vec_others'.
Created version '1' of model 'doc2vec_others'.


on doit apprendre le modèle


2025/03/11 04:09:08 INFO mlflow.models.signature: Inferring model signature from type hints
Registered model 'doc2vec_readme' already exists. Creating a new version of this model...
Created version '2' of model 'doc2vec_readme'.
2025/03/11 04:09:13 INFO mlflow.models.signature: Inferring model signature from type hints
Registered model 'doc2vec_others' already exists. Creating a new version of this model...
Created version '2' of model 'doc2vec_others'.
/home/choux/Documents/M2/Projet_mlops_ingestion/GitMatch/model/Doc2VecTrainer.py:114: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['readme_vectors'] = test_df['readme_preproc'].apply(self._doc2vec_readme.infer_vector)
/home/choux/Documents/M2/Projet_mlops_ingestion/GitMatch/model/Doc2VecT

##### Deuxième étape: inférence des répertoires qui n'ont pas leurs quatre représentations

In [9]:
def ensure_every_repo_is_vectorised(
    database_manager,
    staging_model,
    production_model
):
    """
    Pour le staging et la production on regarde si des répertoires manquent d'une ou plusieurs représentations vectorielles
    On infère leurs représentations vectorielles dans le cas échéant
    """
    # Inférence des nouvelles données
    # Récupération des repos qui n'ont pas encore toutes leurs représentations vectorielles
    df_repos_features_to_predict = database_manager.get_repos_without_all_vectors()

    # Si des répertoires ont besoin d'être inféré on infère
    if len(df_repos_features_to_predict) > 0:
        oriented_df_repos_features_to_predict = df_repos_features_to_predict.to_dict(orient='records')

        stg_vectors = []
        prd_vectors = []
        
        for features in oriented_df_repos_features_to_predict:
            stg_vectors.append(staging_model.predict(features))
            prd_vectors.append(production_model.predict(features))

        vectors = []
        
        for ii, _ in enumerate(stg_vectors):
            vectors_dict = {
                'id': stg_vectors[ii]['id'],
                'staging_readme_vector': stg_vectors[ii]['staging_readme_vector'],
                'staging_others_vector': stg_vectors[ii]['staging_others_vector'],
                'production_readme_vector': prd_vectors[ii]['production_readme_vector'],
                'production_others_vector': prd_vectors[ii]['production_others_vector'],
            }
        
            vectors.append(vectors_dict)

        cols = [
            'id',
            'staging_readme_vector',
            'staging_others_vector',
            'production_readme_vector',
            'production_others_vector',
        ]
        
        df_vectors_to_insert = pd.DataFrame(
            vectors,
            columns=cols
        )

        database_manager.insert_repos_vector_for_both_stages(df_vectors_to_insert)

In [10]:
# On regarde le nombre de répertoires auxquels ils manquent au moins un vecteur
df_repos_without_all_vectors = database_manager.get_repos_without_all_vectors()
print(f'Nombre de répertoires sans ses 4 représentations vectorielles: {len(df_repos_without_all_vectors)}')

# On infère ces répertoires
print('Inférence des répertoires sans leurs 4 représentations vectorielles')
ensure_every_repo_is_vectorised(database_manager, staging_model, production_model)

# On regarde à nouveau le nombre
df_repos_without_all_vectors = database_manager.get_repos_without_all_vectors()
print(f'Nombre de répertoires sans ses 4 représentations vectorielles: : {len(df_repos_without_all_vectors)}')

Nombre de répertoires sans ses 4 représentations vectorielles: 2395
Inférence des répertoires sans leurs 4 représentations vectorilles
Nombre de répertoires sans ses 4 représentations vectorielles: : 0


##### Troisième étape: évaluation des modèles et comparaison des modèles

In [11]:
def compare_models(
    staging_model,
    production_model,
    database_manager,
):
    """
    Évalue les deux modèles (staging et production)
    Compare les résultats
    Retourne le nombre d'évaluations où le modèle de staging a été meilleur
    """
    # Evaluation des deux modèles
    evaluation = Evaluation(staging_model, production_model)
    
    df_vectors = database_manager.get_df_vectors()
    staging_vectors, production_vectors = database_manager.get_df_split_staging_production(df_vectors)

    users_random_chosen_vectors, users_remaining_test_vectors = database_manager.get_users_test_vectors_splitted()

    stg_scores, prd_scores = evaluation.evaluate(
        staging_vectors=staging_vectors,
        production_vectors=production_vectors,
        test_chosen_vectors=users_random_chosen_vectors,
        test_remaining_vectors=users_remaining_test_vectors,
        k=10
    )

    comparison_score = stg_scores > prd_scores
    nb_better_stg_scores = np.count_nonzero(comparison_score)

    return nb_better_stg_scores

In [12]:
nb_better_stg_scores = compare_models(staging_model, production_model, database_manager)
print(f'Le modèle de staging a {nb_better_stg_scores} meilleurs scores')

/home/choux/Documents/M2/Projet_mlops_ingestion/GitMatch/model/Doc2VecModel.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_vectors['similarity_score_with_readme'] = df_vectors[f'{staging}_readme_vector'].apply(lambda vector: compute_similarity_cosinus(vector, readme_vector) if len(vector) > 0 else 0)
/home/choux/Documents/M2/Projet_mlops_ingestion/GitMatch/model/Doc2VecModel.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_vectors['similarity_score_with_readme'] = df_vectors[f'{staging}_read

Le modèle de staging a 2 meilleurs scores


##### Cinquième étape: prise de décision en fonction de la comparaison

In [13]:
def take_action_according_to_comparison_score(
    nb_better_stg_scores,
    database_manager,
):
    """
    Prend une décision en fonction du score issu de l'évaluation
    """
    mlflow_manager = MLFlowManager()
    
    if nb_better_stg_scores > 2:
        # Archive modèle en production
        print('Archivage production')
        mlflow_manager.archive_model_from_production("doc2vec_readme")
        mlflow_manager.archive_model_from_production("doc2vec_others")

        print('Promotion staging')
        # Promut le modèle en staging en production
        mlflow_manager.promote_to_production("doc2vec_readme")
        mlflow_manager.promote_to_production("doc2vec_others")

        print('Passage des vecteurs de staging en production')
        # Transfère les vecteurs de staging en production
        database_manager.upgrade_staging_vectors()
        
        # Requêter pour que l'api mette à jour son modèle et ses vecteurs
        
    else:
        print('Archivage staging')
        # Archive modèle en staging
        mlflow_manager.archive_model_from_staging("doc2vec_readme")
        mlflow_manager.archive_model_from_staging("doc2vec_others")

    print('Apprentissage staging')
    # Apprend un nouveau modèle de staging
    doc2vec_trainer = Doc2VecTrainer(staging='staging')
    train_df, test_df = database_manager.get_train_test_split_features()
    train_vectors, test_vectors = (
        doc2vec_trainer.train(train_df, test_df)
    )

    print('Enregistrement des nouveaux vecteurs de staging')
    # Enregistre les nouveaux vecteurs en staging
    database_manager.insert_repos_vectors_for_one_stage(
        train_vectors,
        'staging'
    )
    database_manager.insert_repos_vectors_for_one_stage(
        test_vectors, 
        'staging'
    )
    

In [14]:
take_action_according_to_comparison_score(nb_better_stg_scores, database_manager)

Archivage staging
Apprentissage staging


2025/03/11 04:20:14 INFO mlflow.models.signature: Inferring model signature from type hints
Registered model 'doc2vec_readme' already exists. Creating a new version of this model...
Created version '3' of model 'doc2vec_readme'.
2025/03/11 04:20:20 INFO mlflow.models.signature: Inferring model signature from type hints
Registered model 'doc2vec_others' already exists. Creating a new version of this model...
Created version '3' of model 'doc2vec_others'.


Enregistrement des nouveaux vecteurs de staging


##### Sixième étape: mise à jour des modèles utilisés par l'api

In [15]:
def update_api():
    """
    Met à jour l'api: modèle en production / vecteurs utilisés pour l'inférence
    """
    base_url = "http://localhost:8000"
    endpoint_for_maj_production = f'{base_url}/update'
    response = requests.post(endpoint_for_maj_production)
    print(response.json())

In [16]:
update_api()

Mise à jour réussie
